In [1]:
import sqlite3
import pandas as pd
import os

In [3]:
LEGACY_DB  = "data/legacy.db"
EXPORT_DIR = "data/exports"
os.makedirs(EXPORT_DIR, exist_ok=True)
 
conn = sqlite3.connect(LEGACY_DB)

In [4]:
tables = ["customers", "accounts", "billing_cycles", "payments"]
structural_results = []
 
for table in tables:
    df = pd.read_sql(f"SELECT * FROM {table}", conn)
    total_rows = len(df)
 
    for col in df.columns:
        null_count = df[col].isna().sum() + (df[col] == "").sum()
        completeness = round((1 - null_count / total_rows) * 100, 2) if total_rows > 0 else 0
        unique_vals  = df[col].nunique()
        structural_results.append({
            "table":        table,
            "column":       col,
            "total_rows":   total_rows,
            "null_count":   null_count,
            "completeness_pct": completeness,
            "unique_values": unique_vals
        })
 
    print(f"  {table}: {total_rows:,} rows")
 
df_structural = pd.DataFrame(structural_results)
df_structural.to_csv(f"{EXPORT_DIR}/profile_structural.csv", index=False)

  customers: 50,000 rows
  accounts: 50,000 rows
  billing_cycles: 500,000 rows
  payments: 449,773 rows


In [5]:
semantic_results = []
 
# 2a. Billing amount range
q = """
    SELECT
        MIN(amount_due)  AS min_amount,
        MAX(amount_due)  AS max_amount,
        AVG(amount_due)  AS avg_amount,
        COUNT(*)         AS total_cycles,
        SUM(CASE WHEN amount_due <= 0 THEN 1 ELSE 0 END) AS negative_or_zero_amounts
    FROM billing_cycles
"""
df_amt = pd.read_sql(q, conn)
print("  Billing amount stats:")
print(df_amt.to_string(index=False))
semantic_results.append({"check": "negative_or_zero_amounts", "value": int(df_amt["negative_or_zero_amounts"][0])})
 
# 2b. Date format check — legacy uses DD/MM/YYYY
# Any date that doesn't match this pattern is a flag
q2 = """
    SELECT COUNT(*) AS malformed_dates
    FROM billing_cycles
    WHERE cycle_start NOT LIKE '__/__/____'
       OR cycle_end   NOT LIKE '__/__/____'
"""
df_dates = pd.read_sql(q2, conn)
print(f"\n  Malformed date entries: {int(df_dates['malformed_dates'][0]):,}")
semantic_results.append({"check": "malformed_dates", "value": int(df_dates["malformed_dates"][0])})
 
# 2c. VAT rate consistency
q3 = """
    SELECT DISTINCT vat_rate, COUNT(*) as count
    FROM billing_cycles
    GROUP BY vat_rate
    ORDER BY count DESC
"""
df_vat = pd.read_sql(q3, conn)
print(f"\n  VAT rate distribution:")
print(df_vat.to_string(index=False))
 
# 2d. Payment amounts vs amount due
q4 = """
    SELECT
        COUNT(*) AS overpaid_records
    FROM payments p
    JOIN billing_cycles b ON p.cycle_id = b.cycle_id
    WHERE p.amount_paid > b.amount_due * 1.01
"""
df_overpay = pd.read_sql(q4, conn)
print(f"\n  Overpaid records: {int(df_overpay['overpaid_records'][0]):,}")
semantic_results.append({"check": "overpaid_records", "value": int(df_overpay["overpaid_records"][0])})
 
pd.DataFrame(semantic_results).to_csv(f"{EXPORT_DIR}/profile_semantic.csv", index=False)

  Billing amount stats:
 min_amount  max_amount  avg_amount  total_cycles  negative_or_zero_amounts
      28.49      457.46  185.924008        500000                         0

  Malformed date entries: 0

  VAT rate distribution:
 vat_rate  count
    0.135 500000

  Overpaid records: 0


In [6]:
relational_results = []
 
# 3a. Accounts with no customer reference
q5 = """
    SELECT COUNT(*) AS orphaned_accounts
    FROM accounts a
    LEFT JOIN customers c ON a.customer_id = c.customer_id
    WHERE c.customer_id IS NULL
"""
df_orph = pd.read_sql(q5, conn)
val = int(df_orph["orphaned_accounts"][0])
print(f"  Orphaned accounts (no customer): {val:,}")
relational_results.append({"check": "orphaned_accounts", "value": val})
 
# 3b. Billing cycles with no account
q6 = """
    SELECT COUNT(*) AS orphaned_cycles
    FROM billing_cycles bc
    LEFT JOIN accounts a ON bc.account_id = a.account_id
    WHERE a.account_id IS NULL
"""
df_orph2 = pd.read_sql(q6, conn)
val2 = int(df_orph2["orphaned_cycles"][0])
print(f"  Orphaned billing cycles (no account): {val2:,}")
relational_results.append({"check": "orphaned_cycles", "value": val2})
 
# 3c. Payments referencing non-existent cycles
q7 = """
    SELECT COUNT(*) AS orphaned_payments
    FROM payments p
    LEFT JOIN billing_cycles bc ON p.cycle_id = bc.cycle_id
    WHERE bc.cycle_id IS NULL
"""
df_orph3 = pd.read_sql(q7, conn)
val3 = int(df_orph3["orphaned_payments"][0])
print(f"  Orphaned payments (no billing cycle): {val3:,}")
relational_results.append({"check": "orphaned_payments", "value": val3})
 
pd.DataFrame(relational_results).to_csv(f"{EXPORT_DIR}/profile_relational.csv", index=False)

  Orphaned accounts (no customer): 0
  Orphaned billing cycles (no account): 0
  Orphaned payments (no billing cycle): 0


In [7]:
q_summary = """
    SELECT
        c.customer_id,
        c.region,
        c.fuel_type,
        c.tariff_plan,
        c.payment_method,
        c.is_active,
        a.account_id,
        a.account_type,
        a.current_balance,
        COUNT(bc.cycle_id)          AS total_billing_cycles,
        SUM(bc.amount_due)          AS total_billed,
        COALESCE(SUM(p.amount_paid), 0) AS total_paid,
        SUM(bc.amount_due) - COALESCE(SUM(p.amount_paid), 0) AS outstanding_balance,
        SUM(CASE WHEN bc.status = 'Overdue' THEN 1 ELSE 0 END) AS overdue_cycles
    FROM customers c
    JOIN accounts a         ON c.customer_id  = a.customer_id
    JOIN billing_cycles bc  ON a.account_id   = bc.account_id
    LEFT JOIN payments p    ON bc.cycle_id     = p.cycle_id
    GROUP BY c.customer_id, a.account_id
"""
df_summary = pd.read_sql(q_summary, conn)
df_summary.to_csv(f"{EXPORT_DIR}/account_summary_legacy.csv", index=False)
print(f"  Account summary exported: {len(df_summary):,} rows")
print(f"  → Exported: account_summary_legacy.csv")
 
conn.close()

  Account summary exported: 50,000 rows
  → Exported: account_summary_legacy.csv
